# **Cálculo de la valoración (*rating*) y el potencial de un jugador**

En este estudio se calcula, a partir de los datos de cada jugador en cada partido, una **valoración (*rating*)**. El proceso parte de una nota por partido (0–10) ajustada por el contexto, la agrega en el tiempo y termina produciendo dos números tipo videojuego: un **Rating** actual (65–95) y un **Potential** (65–99).

El cálculo tiene en cuenta el **Elo del club y del rival**, los **minutos jugados**, el **resultado** y la **importancia de la competición**.

In [6]:
import pandas as pd
import os
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import warnings

warnings.filterwarnings("ignore", message="Parsing dates in %d/%m/%Y format when dayfirst=False")

DATA_PATH = r"d:\data"

# Dataframes necesarios
PLAYER = pd.read_csv(os.path.join(DATA_PATH, "silver", "cleaning_2", "player_clean_2.csv"), sep=";")
STATS_PLAYER = pd.read_csv(os.path.join(DATA_PATH, "silver", "cleaning_3", "player_stats_clean_3_2.csv"), sep=";")
MATCH = pd.read_csv(os.path.join(DATA_PATH, "silver", "cleaning_2", "match_clean_2.csv"), sep=";")
TEAM = pd.read_csv(os.path.join(DATA_PATH, "silver", "cleaning_2", "team_clean_2.csv"), sep=";")

---

## **1. Preparación de los datos**

- **Datos de partido.** A cada partido se le asigna el peso de competición (`TournWeight`, el mismo esquema que el Elo: 20–50). Se reestructura a una fila por equipo y partido, y se calculan la **diferencia de goles** (`ScoreDiff = Score − OppScore`) y el **resultado** (1 victoria / 0 empate / −1 derrota).
- **Datos de jugador.** Se cruzan las estadísticas de cada jugador con el contexto del partido (Elo propio y rival, diferencia de goles, minutos). Se separan en un diccionario **por posición** (`GK`, `FB`, `CB`, `DM`, `AM`, `WG`, `ST`), porque cada posición se valora con criterios distintos.

In [8]:
# --------------------------------------------------------------------------------------
# PREPARACIÓN DE DATOS DE PARTIDOS
# --------------------------------------------------------------------------------------
def prepare_match_data(match_dataframe: pd.DataFrame) -> pd.DataFrame:

    match_df = match_dataframe.copy()

    # Añadimos peso de tournament
    tournament_k = {"la_liga": 40, "premier_league": 40, "bundesliga": 40, "serie_a": 40, "ligue_1": 40, "champions_league": 50, "europa_league": 30, "copa_del_rey": 30, "la_liga_hypermotion": 20,
                    "championship": 20, "eredivise": 30, "liga_portugal": 30, "conference_league": 30, "copa_libertadores": 20, "liga_mx": 20, "serie_a_brazil": 20, "liga_profesional": 20,
                    "first_division_a": 20, "major_league_soccer": 20, "saudi_pro_league": 20, "super_lig": 20, "superligaen": 20, "swiss_super_league": 20, "allvenskan": 20, "eliteserien": 20,
                    "conmebol_sudamericana": 20}
    match_df["TournamentWeight"] = match_df["League"].map(tournament_k)

    # Selección datos por equipo local y visitante
    match_df_home = match_df[["ID", "HomeTeam", "TournamentWeight", "HomeScore", "AwayScore", "HomeElo", "AwayElo"]]
    match_df_away = match_df[["ID", "AwayTeam", "TournamentWeight", "AwayScore", "HomeScore", "AwayElo", "HomeElo"]]

    # Nombres de columnas y concatenación
    match_df_home.columns = ["Match", "Team", "TournWeight", "Score", "OppScore", "Elo", "OppElo"]
    match_df_away.columns = ["Match", "Team", "TournWeight", "Score", "OppScore", "Elo", "OppElo"]
    match_df = pd.concat([match_df_home, match_df_away])

    # Añadimos diferencia de goles y si es victoria, empate o derrota
    match_df["ScoreDiff"] = match_df["Score"] - match_df["OppScore"]
    match_df["Result"] = np.where(match_df["ScoreDiff"] > 0, 1, np.where(match_df["ScoreDiff"] < 0, -1, 0))             # Victoria 1, derrota -1, empate 0

    return match_df

# --------------------------------------------------------------------------------------
# PREPARACIÓN DE DATOS DE JUGADORES (ESTADISTICAS)
# --------------------------------------------------------------------------------------
def prepare_stats_player(stats_dataframe: pd.DataFrame, match_dataframe: pd.DataFrame) -> dict:

    # Obtenemos los datos 
    stats_df = stats_dataframe.copy()
    match_df = prepare_match_data(match_dataframe=match_dataframe.copy())

    # Merge
    df = match_df.merge(stats_df, on=["Match", "Team"], how="inner")

    # Borramos columnas y sacamos valores nulos en jugador
    df = df.drop(columns=["ShirtNumber", "Team"]).dropna(subset="Player")

    # Diccionario para añadir informacion
    output_dict = {}

    # Para cada posición, buscamos jugadores
    for pos in ["GK", "FB", "CB", "DM", "AM", "WG", "ST"]:
        if pos == "FB":
            pos_df = df[df["Position"].isin(["RB", "LB"])].drop(columns="Position")
        elif pos == "WG":
            pos_df = df[df["Position"].isin(["RW", "LW"])].drop(columns="Position")
        else:
            pos_df = df[df["Position"]==pos].drop(columns="Position")

        # Ponemos jugador al principio e indexamos al diccionario
        pos_df = pos_df[["Player"] + [col for col in pos_df.columns if col != "Player"]]
        output_dict[pos] = pos_df.sort_values(by=["Player"]).reset_index(drop=True)

    return output_dict

In [9]:
dict_positions_df = prepare_stats_player(stats_dataframe=STATS_PLAYER.copy(), match_dataframe=MATCH.copy())

---

## **2. Rating del partido según la posición**

Para cada posición existe una función específica que calcula la nota del partido. Todas siguen la **misma estructura**, pero con métricas y pesos propios.

### **2.1. Normalización de métricas**

Las estadísticas de volumen se llevan a $[0, 1]$ con una normalización *min–max* acotada, definiendo a mano el máximo razonable de cada métrica:

$$
\text{norm}(x) = \mathrm{clip}\!\left(\frac{x - \min}{\max - \min},\; 0,\; 1\right)
$$

Por ejemplo, para un portero `SavedShotsInsideBox` se normaliza con $\min=0,\ \max=12$. Las métricas que ya son tasas o porcentajes (p. ej. `SaveRate`) entran directamente.

### **2.2. Estructura del rating**

La nota se construye sobre una **base de 6.0** a la que se suman bonificaciones y se restan penalizaciones:

$$
\text{base} = 6.0 + \text{Positive} + \text{Negative}
$$

- **Positive** = suma ponderada de métricas **buenas** normalizadas. Los pesos dependen de la posición y reflejan qué es importante en cada rol. Por ejemplo, un portero suma sobre todo por `SaveRate` (0.75), `GoalsPrevented` (0.55) o `HighClaimRate` (0.45); un delantero suma por `Goals` (0.64), `ExpectedGoalsOnTarget` (0.40) o `ShotAccuracy` (0.40).
- **Negative** = penalizaciones comunes a casi todas las posiciones: tarjeta amarilla ($-1.5$), roja ($-4.0$), errores que llevan a tiro/gol, goles en propia, penaltis fallados, etc.

### **2.3. Ajustes contextuales**

La base se multiplica por tres modificadores que sitúan la nota en su contexto:

$$
\text{EloMod} = 1 + 0.05 \cdot \frac{R_{\text{rival}} - R_{\text{equipo}}}{400}
$$

(premia rendir bien contra rivales más fuertes; ±5 % por cada 400 puntos de Elo de diferencia).

$$
\text{MinMod} = \mathrm{clip}\!\left(\frac{\text{Minutos}}{60},\; 0.5,\; 1\right)
\qquad
\text{ScoreMod} = 1 + 0.1 \cdot \text{ScoreDiff}
$$

(`MinMod` modera la nota de quien juega poco; `ScoreMod` premia/penaliza según la diferencia de goles del equipo).

### **2.4. Rating final del partido**

$$
\text{Rating} = \mathrm{clip}\Big(\;\mathrm{clip}(\text{base},\,0,\,10)\;\cdot\;\text{EloMod}\cdot\text{MinMod}\cdot\text{ScoreMod},\;\; 0,\; 10\Big)
$$

Se calcula para todas las posiciones y se concatena en un único *dataframe* de notas por jugador y partido.

---


In [12]:
# --------------------------------------------------------------------------------------
# NORMALIZACIÓN DE COLUMNAS - a partir de una row de un dataframe, y valores máximos y mínimos, normaliza la columna
# --------------------------------------------------------------------------------------
def normalize_column(series: pd.Series, min_value: float, max_value: float) -> pd.Series:

    # Evitar división por 0
    if max_value == min_value:
        return pd.Series(np.zeros(len(series)), index=series.index)
    normalized = (series - min_value) / (max_value - min_value)

    # Limitar entre 0 y 1
    normalized = normalized.clip(0, 1)

    return normalized

# --------------------------------------------------------------------------------------
# DEFINE RATINGS DE PORTEROS
# --------------------------------------------------------------------------------------
def get_gk_ratings(dict_positions_df: dict) -> pd.DataFrame:

    df = dict_positions_df["GK"].copy()

    # Obtenemos estadísticas
    stats_df = df[["Player","Match","Elo","OppElo","ScoreDiff","MinutesPlayed","SaveRate","GoalsPrevented","SavedShotsInsideBox","HighClaimRate","KeeperSweeperAccuracy",
                   "CrossesNotClaimed","Punches","PassAccuracy","LongBallAccuracy","KeeperSweeperActions","PenaltySaveRate","PenaltyConceded","ErrorsLeadToShot","ErrorsLeadToGoal",
                   "Goals","GoalAssists","YellowCards","RedCards"]]
    stats_df = stats_df.fillna(0)

    # Aplicamos normalización de columnas para aquellas contada
    stats_df["GoalsPrevented"] = normalize_column(series=stats_df["GoalsPrevented"], min_value=-3, max_value=3)
    stats_df["SavedShotsInsideBox"] = normalize_column(series=stats_df["SavedShotsInsideBox"], min_value=0, max_value=12)
    stats_df["CrossesNotClaimed"] = normalize_column(series=stats_df["CrossesNotClaimed"], min_value=0, max_value=3)
    stats_df["Punches"] = normalize_column(series=stats_df["Punches"], min_value=0, max_value=8)
    stats_df["KeeperSweeperActions"] = normalize_column(series=stats_df["KeeperSweeperActions"], min_value=0, max_value=8)
    stats_df["PenaltySaveRate"] = normalize_column(series=stats_df["PenaltySaveRate"], min_value=0, max_value=1.2)
    stats_df["PenaltyConceded"] = normalize_column(series=stats_df["PenaltyConceded"], min_value=0, max_value=1.2)
    stats_df["ErrorsLeadToShot"] = normalize_column(series=stats_df["ErrorsLeadToShot"], min_value=0, max_value=2)
    stats_df["ErrorsLeadToGoal"] = normalize_column(series=stats_df["ErrorsLeadToGoal"], min_value=0, max_value=1)
    stats_df["Goals"] = normalize_column(series=stats_df["Goals"], min_value=0, max_value=3)
    stats_df["GoalAssists"] = normalize_column(series=stats_df["GoalAssists"], min_value=0, max_value=3)
    stats_df["YellowCards"] = normalize_column(series=stats_df["YellowCards"], min_value=0, max_value=1.5)
    stats_df["RedCards"] = normalize_column(series=stats_df["RedCards"], min_value=0, max_value=1)

    # Rating inicial
    stats_df["InitialRating"] = 6.0

    # Bonus positivo
    stats_df["Positive"] = (0.75 * stats_df["SaveRate"] + 0.55 * stats_df["GoalsPrevented"] + 0.35 * stats_df["SavedShotsInsideBox"] +
                            0.45 * stats_df["HighClaimRate"] + 0.30 * stats_df["KeeperSweeperAccuracy"] + 0.20 * stats_df["Punches"] +
                            0.40 * stats_df["PassAccuracy"] + 0.25 * stats_df["LongBallAccuracy"] + 0.15 * stats_df["KeeperSweeperActions"] +
                            0.60 * stats_df["PenaltySaveRate"] +
                            0.30 * stats_df["Goals"] + 0.25 * stats_df["GoalAssists"])

    # Penalizaciones
    stats_df["Negative"] = (-0.40 * stats_df["CrossesNotClaimed"] - 0.30 * stats_df["PenaltyConceded"] - 0.60 * stats_df["ErrorsLeadToShot"] - 1.20 * stats_df["ErrorsLeadToGoal"] -
                            1.50 * stats_df["YellowCards"] - 4.00 * stats_df["RedCards"])

    # Ajuste contextual
    stats_df["EloMod"] = 1 + 0.05*((stats_df["OppElo"] - stats_df["Elo"]) / 400)
    stats_df["MinMod"] = (stats_df["MinutesPlayed"] / 60).clip(0.5, 1)
    stats_df["ScoreMod"] = 1 + 0.1*stats_df["ScoreDiff"]

    # Rating
    stats_df["Rating"] = ((stats_df["InitialRating"] + stats_df["Positive"] + stats_df["Negative"]).clip(0, 10) *
                          stats_df["EloMod"] * stats_df["MinMod"] * stats_df["ScoreMod"]).clip(0, 10)

    stats_df = stats_df[["Player", "Match", "Rating"]]

    return stats_df

# --------------------------------------------------------------------------------------
# DEFINE RATINGS DE LATERALES
# --------------------------------------------------------------------------------------
def get_fb_ratings(dict_positions_df: dict) -> pd.DataFrame:

    df = dict_positions_df["FB"].copy()

    # Obtenemos estadísticas
    stats_df = df[["Player","Match","Elo","OppElo","ScoreDiff","MinutesPlayed","TackleAccuracy","Interceptions","DuelWinRate","BallRecoveries","PassAccuracy","OwnGoals",
                   "OppositionHalfPassAccuracy","ProgressiveFieldTilt","CrossAccuracy","KeyPasses","GoalAssists","WasFouledRate","YellowCards","RedCards","ErrorsLeadToGoal",
                   "Goals"]]
    stats_df = stats_df.fillna(0)

    # Aplicamos normalización de columnas para aquellas contada
    stats_df["Interceptions"] = normalize_column(series=stats_df["Interceptions"], min_value=0, max_value=8)
    stats_df["BallRecoveries"] = normalize_column(series=stats_df["BallRecoveries"], min_value=0, max_value=12)
    stats_df["ProgressiveFieldTilt"] = normalize_column(series=stats_df["ProgressiveFieldTilt"], min_value=0, max_value=10)
    stats_df["KeyPasses"] = normalize_column(series=stats_df["KeyPasses"], min_value=0, max_value=5)
    stats_df["GoalAssists"] = normalize_column(series=stats_df["GoalAssists"], min_value=0, max_value=3)
    stats_df["Goals"] = normalize_column(series=stats_df["Goals"], min_value=0, max_value=3)
    stats_df["YellowCards"] = normalize_column(series=stats_df["YellowCards"], min_value=0, max_value=1.5)
    stats_df["RedCards"] = normalize_column(series=stats_df["RedCards"], min_value=0, max_value=1)
    stats_df["ErrorsLeadToGoal"] = normalize_column(series=stats_df["ErrorsLeadToGoal"], min_value=0, max_value=2)
    stats_df["OwnGoals"] = normalize_column(series=stats_df["OwnGoals"], min_value=0, max_value=2)

    # Rating inicial
    stats_df["InitialRating"] = 6.0

    # Bonus positivo
    stats_df["Positive"] = (0.48 * stats_df["TackleAccuracy"] + 0.32 * stats_df["Interceptions"] + 0.32 * stats_df["DuelWinRate"] + 0.28 * stats_df["BallRecoveries"] +
                            0.48 * stats_df["PassAccuracy"] + 0.40 * stats_df["OppositionHalfPassAccuracy"] + 0.32 * stats_df["ProgressiveFieldTilt"] +
                            0.40 * stats_df["CrossAccuracy"] + 0.32 * stats_df["KeyPasses"] + 0.28 * stats_df["GoalAssists"] +
                            0.40 * stats_df["WasFouledRate"] +
                            0.30 * stats_df["Goals"])

    # Penalizaciones
    stats_df["Negative"] = (-3.00 * stats_df["ErrorsLeadToGoal"] - 2.00 * stats_df["OwnGoals"] -
                            1.50 * stats_df["YellowCards"] - 4.00 * stats_df["RedCards"])

    # Ajuste contextual
    stats_df["EloMod"] = 1 + 0.05*((stats_df["OppElo"] - stats_df["Elo"]) / 400)
    stats_df["MinMod"] = (stats_df["MinutesPlayed"] / 60).clip(0.5, 1)
    stats_df["ScoreMod"] = 1 + 0.1*stats_df["ScoreDiff"]

    # Rating
    stats_df["Rating"] = ((stats_df["InitialRating"] + stats_df["Positive"] + stats_df["Negative"]).clip(0, 10) *
                          stats_df["EloMod"] * stats_df["MinMod"] * stats_df["ScoreMod"]).clip(0, 10)

    stats_df = stats_df[["Player", "Match", "Rating"]]

    return stats_df

# --------------------------------------------------------------------------------------
# DEFINE RATINGS DE DEFENSAS CENTRALES
# --------------------------------------------------------------------------------------
def get_cb_ratings(dict_positions_df: dict) -> pd.DataFrame:

    df = dict_positions_df["CB"].copy()

    # Obtenemos estadísticas
    stats_df = df[["Player","Match","Elo","OppElo","ScoreDiff","MinutesPlayed","DuelWinRate","AerialWinRate","TackleAccuracy","Interceptions","Clearances","OutfielderBlocks",
                   "LastManTackles","PassAccuracy","LongBallAccuracy","OwnHalfPassAccuracy","ErrorsLeadToShot","ErrorsLeadToGoal","OwnGoals","ClearanceOffLine",
                   "Goals","GoalAssists","YellowCards","RedCards"]]
    stats_df = stats_df.fillna(0)

    # Aplicamos normalización de columnas para aquellas contada
    stats_df["Interceptions"] = normalize_column(series=stats_df["Interceptions"], min_value=0, max_value=8)
    stats_df["Clearances"] = normalize_column(series=stats_df["Clearances"], min_value=0, max_value=8)
    stats_df["OutfielderBlocks"] = normalize_column(series=stats_df["OutfielderBlocks"], min_value=0, max_value=5)
    stats_df["LastManTackles"] = normalize_column(series=stats_df["LastManTackles"], min_value=0, max_value=2)
    stats_df["ErrorsLeadToShot"] = normalize_column(series=stats_df["ErrorsLeadToShot"], min_value=0, max_value=2)
    stats_df["ErrorsLeadToGoal"] = normalize_column(series=stats_df["ErrorsLeadToGoal"], min_value=0, max_value=1)
    stats_df["OwnGoals"] = normalize_column(series=stats_df["OwnGoals"], min_value=0, max_value=2)
    stats_df["ClearanceOffLine"] = normalize_column(series=stats_df["ClearanceOffLine"], min_value=0, max_value=2)
    stats_df["Goals"] = normalize_column(series=stats_df["Goals"], min_value=0, max_value=3)
    stats_df["GoalAssists"] = normalize_column(series=stats_df["GoalAssists"], min_value=0, max_value=3)
    stats_df["YellowCards"] = normalize_column(series=stats_df["YellowCards"], min_value=0, max_value=1.5)
    stats_df["RedCards"] = normalize_column(series=stats_df["RedCards"], min_value=0, max_value=1)

    # Rating inicial
    stats_df["InitialRating"] = 6.0

    # Bonus positivo
    stats_df["Positive"] = (0.48 * stats_df["DuelWinRate"] + 0.48 * stats_df["AerialWinRate"] + 0.32 * stats_df["TackleAccuracy"] + 0.32 * stats_df["Interceptions"] +
                            0.40 * stats_df["Clearances"] + 0.32 * stats_df["OutfielderBlocks"] + 0.28 * stats_df["LastManTackles"] +
                            0.48 * stats_df["PassAccuracy"] + 0.32 * stats_df["LongBallAccuracy"] + 0.20 * stats_df["OwnHalfPassAccuracy"] +
                            1.50 * stats_df["ClearanceOffLine"] +
                            0.30 * stats_df["Goals"] + 0.25 * stats_df["GoalAssists"])

    # Penalizaciones
    stats_df["Negative"] = (-1.50 * stats_df["ErrorsLeadToShot"] - 4.00 * stats_df["ErrorsLeadToGoal"] - 2.00 * stats_df["OwnGoals"] -
                            1.50 * stats_df["YellowCards"] - 4.00 * stats_df["RedCards"])

    # Ajuste contextual
    stats_df["EloMod"] = 1 + 0.05*((stats_df["OppElo"] - stats_df["Elo"]) / 400)
    stats_df["MinMod"] = (stats_df["MinutesPlayed"] / 60).clip(0.5, 1)
    stats_df["ScoreMod"] = 1 + 0.1*stats_df["ScoreDiff"]

    # Rating
    stats_df["Rating"] = ((stats_df["InitialRating"] + stats_df["Positive"] + stats_df["Negative"]).clip(0, 10) *
                          stats_df["EloMod"] * stats_df["MinMod"] * stats_df["ScoreMod"]).clip(0, 10)

    stats_df = stats_df[["Player", "Match", "Rating"]]

    return stats_df

# --------------------------------------------------------------------------------------
# DEFINE RATINGS DE MEDIOCENTROS DEFENSIVOS
# --------------------------------------------------------------------------------------
def get_dm_ratings(dict_positions_df: dict) -> pd.DataFrame:

    df = dict_positions_df["DM"].copy()

    # Obtenemos estadísticas
    stats_df = df[["Player","Match","Elo","OppElo","ScoreDiff","MinutesPlayed","Interceptions","BallRecoveries","TackleAccuracy","DefensiveActionSuccess",
                   "PassAccuracy","OwnHalfPassAccuracy","PossessionLossRate","ProgressiveFieldTilt","LongBallAccuracy","KeyPasses","ExpectedAssists",
                   "DuelWinRate","AerialWinRate","YellowCards","RedCards","Goals","GoalAssists"]]
    stats_df = stats_df.fillna(0)

    # Aplicamos normalización de columnas para aquellas contada
    stats_df["Interceptions"] = normalize_column(series=stats_df["Interceptions"], min_value=0, max_value=8)
    stats_df["BallRecoveries"] = normalize_column(series=stats_df["BallRecoveries"], min_value=0, max_value=12)
    stats_df["ProgressiveFieldTilt"] = normalize_column(series=stats_df["ProgressiveFieldTilt"], min_value=0, max_value=10)
    stats_df["KeyPasses"] = normalize_column(series=stats_df["KeyPasses"], min_value=0, max_value=5)
    stats_df["ExpectedAssists"] = normalize_column(series=stats_df["ExpectedAssists"], min_value=0, max_value=1)
    stats_df["Goals"] = normalize_column(series=stats_df["Goals"], min_value=0, max_value=3)
    stats_df["GoalAssists"] = normalize_column(series=stats_df["GoalAssists"], min_value=0, max_value=3)
    stats_df["YellowCards"] = normalize_column(series=stats_df["YellowCards"], min_value=0, max_value=1.5)
    stats_df["RedCards"] = normalize_column(series=stats_df["RedCards"], min_value=0, max_value=1)

    # Rating inicial
    stats_df["InitialRating"] = 6.0

    # Bonus positivo
    stats_df["Positive"] = (0.48 * stats_df["Interceptions"] + 0.40 * stats_df["BallRecoveries"] + 0.32 * stats_df["TackleAccuracy"] + 0.20 * stats_df["DefensiveActionSuccess"] +
                            0.48 * stats_df["PassAccuracy"] + 0.32 * stats_df["OwnHalfPassAccuracy"] - 0.32 * stats_df["PossessionLossRate"] + 0.32 * stats_df["ProgressiveFieldTilt"] + 0.16 * stats_df["LongBallAccuracy"] +
                            0.32 * stats_df["KeyPasses"] + 0.28 * stats_df["ExpectedAssists"] +
                            0.24 * stats_df["DuelWinRate"] + 0.16 * stats_df["AerialWinRate"] +
                            0.30 * stats_df["Goals"] + 0.25 * stats_df["GoalAssists"])

    # Penalizaciones
    stats_df["Negative"] = (-1.50 * stats_df["YellowCards"] - 4.00 * stats_df["RedCards"])

    # Ajuste contextual
    stats_df["EloMod"] = 1 + 0.05*((stats_df["OppElo"] - stats_df["Elo"]) / 400)
    stats_df["MinMod"] = (stats_df["MinutesPlayed"] / 60).clip(0.5, 1)
    stats_df["ScoreMod"] = 1 + 0.1*stats_df["ScoreDiff"]

    # Rating
    stats_df["Rating"] = ((stats_df["InitialRating"] + stats_df["Positive"] + stats_df["Negative"]).clip(0, 10) *
                          stats_df["EloMod"] * stats_df["MinMod"] * stats_df["ScoreMod"]).clip(0, 10)

    stats_df = stats_df[["Player", "Match", "Rating"]]

    return stats_df

# --------------------------------------------------------------------------------------
# DEFINE RATINGS DE MEDIOCENTROS OFENSIVOS
# --------------------------------------------------------------------------------------
def get_am_ratings(dict_positions_df: dict) -> pd.DataFrame:

    df = dict_positions_df["AM"].copy()

    # Obtenemos estadísticas
    stats_df = df[["Player","Match","Elo","OppElo","ScoreDiff","MinutesPlayed","KeyPasses","ExpectedAssists","GoalAssists","BigChancesCreated",
                   "Goals","ExpectedGoals","ShotsOnTarget","BigChancesMissed","PassAccuracy","OppositionHalfPassAccuracy","PossessionLossRate",
                   "Touches","ContestWinRate","YellowCards","RedCards"]]
    stats_df = stats_df.fillna(0)

    # Aplicamos normalización de columnas para aquellas contada
    stats_df["KeyPasses"] = normalize_column(series=stats_df["KeyPasses"], min_value=0, max_value=5)
    stats_df["ExpectedAssists"] = normalize_column(series=stats_df["ExpectedAssists"], min_value=0, max_value=1)
    stats_df["GoalAssists"] = normalize_column(series=stats_df["GoalAssists"], min_value=0, max_value=3)
    stats_df["BigChancesCreated"] = normalize_column(series=stats_df["BigChancesCreated"], min_value=0, max_value=4)
    stats_df["Goals"] = normalize_column(series=stats_df["Goals"], min_value=0, max_value=3)
    stats_df["ExpectedGoals"] = normalize_column(series=stats_df["ExpectedGoals"], min_value=0, max_value=1.5)
    stats_df["ShotsOnTarget"] = normalize_column(series=stats_df["ShotsOnTarget"], min_value=0, max_value=5)
    stats_df["BigChancesMissed"] = normalize_column(series=stats_df["BigChancesMissed"], min_value=0, max_value=3)
    stats_df["Touches"] = normalize_column(series=stats_df["Touches"], min_value=0, max_value=100)
    stats_df["YellowCards"] = normalize_column(series=stats_df["YellowCards"], min_value=0, max_value=1.5)
    stats_df["RedCards"] = normalize_column(series=stats_df["RedCards"], min_value=0, max_value=1)

    # Rating inicial
    stats_df["InitialRating"] = 6.0

    # Bonus positivo
    stats_df["Positive"] = (0.48 * stats_df["KeyPasses"] + 0.40 * stats_df["ExpectedAssists"] + 0.40 * stats_df["GoalAssists"] + 0.32 * stats_df["BigChancesCreated"] +              # Creación de juego: max +1.60
                            0.48 * stats_df["Goals"] + 0.32 * stats_df["ExpectedGoals"] + 0.24 * stats_df["ShotsOnTarget"] +                                                             # Contribución al gol: max +1.04
                            0.40 * stats_df["PassAccuracy"] + 0.32 * stats_df["OppositionHalfPassAccuracy"] - 0.08 * stats_df["PossessionLossRate"] +                                    # Circulación: max +0.72
                            0.20 * stats_df["Touches"] + 0.20 * stats_df["ContestWinRate"])                                                                                              # Participación: max +0.40

    # Penalizaciones
    stats_df["Negative"] = (-0.16 * stats_df["BigChancesMissed"] - 1.50 * stats_df["YellowCards"] - 4.00 * stats_df["RedCards"])

    # Ajuste por nivel del rival - ajuste +-5% por cada 400 puntos de diferencia
    stats_df["EloMod"] = 1 + 0.05*((stats_df["OppElo"] - stats_df["Elo"]) / 400)
    stats_df["MinMod"] = (stats_df["MinutesPlayed"] / 60).clip(0.5, 1)
    stats_df["ScoreMod"] = 1 + 0.1*stats_df["ScoreDiff"]

    # Cálculo del rating
    stats_df["Rating"] = ((stats_df["InitialRating"] + stats_df["Positive"] + stats_df["Negative"]).clip(0, 10) * stats_df["EloMod"] * stats_df["MinMod"] * stats_df["ScoreMod"]).clip(0, 10)

    # Selección de las columnas a mostrar
    stats_df = stats_df[["Player", "Match", "Rating"]]

    return stats_df

# --------------------------------------------------------------------------------------
# DEFINE RATINGS DE EXTREMOS
# --------------------------------------------------------------------------------------
def get_wg_ratings(dict_positions_df: dict) -> pd.DataFrame:

    df = dict_positions_df["WG"].copy()

    # Obtenemos estadísticas
    stats_df = df[["Player","Match","Elo","OppElo","ScoreDiff","MinutesPlayed","CrossAccuracy","ContestWinRate","AccurateCrosses",
                   "Goals","GoalAssists","ExpectedGoals","ExpectedAssists","KeyPasses","BigChancesCreated",
                   "BallRecoveries","WasFouled","DefensiveActions","PassAccuracy","PossessionLossRate",
                   "OppositionHalfPassShare","YellowCards","RedCards"]]
    stats_df = stats_df.fillna(0)

    # Aplicamos normalización de columnas para aquellas contada
    stats_df["AccurateCrosses"] = normalize_column(series=stats_df["AccurateCrosses"], min_value=0, max_value=8)
    stats_df["Goals"] = normalize_column(series=stats_df["Goals"], min_value=0, max_value=3)
    stats_df["GoalAssists"] = normalize_column(series=stats_df["GoalAssists"], min_value=0, max_value=3)
    stats_df["ExpectedGoals"] = normalize_column(series=stats_df["ExpectedGoals"], min_value=0, max_value=1.5)
    stats_df["ExpectedAssists"] = normalize_column(series=stats_df["ExpectedAssists"], min_value=0, max_value=1.5)
    stats_df["KeyPasses"] = normalize_column(series=stats_df["KeyPasses"], min_value=0, max_value=5)
    stats_df["BigChancesCreated"] = normalize_column(series=stats_df["BigChancesCreated"], min_value=0, max_value=5)
    stats_df["BallRecoveries"] = normalize_column(series=stats_df["BallRecoveries"], min_value=0, max_value=12)
    stats_df["WasFouled"] = normalize_column(series=stats_df["WasFouled"], min_value=0, max_value=8)
    stats_df["DefensiveActions"] = normalize_column(series=stats_df["DefensiveActions"], min_value=0, max_value=10)
    stats_df["YellowCards"] = normalize_column(series=stats_df["YellowCards"], min_value=0, max_value=1.5)
    stats_df["RedCards"] = normalize_column(series=stats_df["RedCards"], min_value=0, max_value=1)

    # Rating inicial
    stats_df["InitialRating"] = 6.0

    # Bonus positivo
    stats_df["Positive"] = (0.48 * stats_df["CrossAccuracy"] + 0.40 * stats_df["ContestWinRate"] + 0.32 * stats_df["AccurateCrosses"] +                                      # Desborde y centros: max +1.20
                            0.48 * stats_df["Goals"] + 0.40 * stats_df["GoalAssists"] + 0.32 * (stats_df["ExpectedGoals"] + stats_df["ExpectedAssists"]) + 0.20 * stats_df["KeyPasses"] +   # Contribución al gol: max +1.40
                            0.32 * stats_df["BallRecoveries"] + 0.24 * stats_df["WasFouled"] + 0.24 * stats_df["DefensiveActions"] +                                          # Presión y recuperación: max +0.80
                            0.32 * stats_df["PassAccuracy"] - 0.20 * stats_df["PossessionLossRate"] + 0.12 * stats_df["OppositionHalfPassShare"] +                          # Mantenimiento balón: max +0.24
                            1.00 * stats_df["BigChancesCreated"])                                                                                                               # Bonus creativo

    # Penalizaciones
    stats_df["Negative"] = (-1.50 * stats_df["YellowCards"] - 4.00 * stats_df["RedCards"])

    # Ajuste por nivel del rival - ajuste +-5% por cada 400 puntos de diferencia
    stats_df["EloMod"] = 1 + 0.05*((stats_df["OppElo"] - stats_df["Elo"]) / 400)
    stats_df["MinMod"] = (stats_df["MinutesPlayed"] / 60).clip(0.5, 1)
    stats_df["ScoreMod"] = 1 + 0.1*stats_df["ScoreDiff"]

    # Cálculo del rating
    stats_df["Rating"] = ((stats_df["InitialRating"] + stats_df["Positive"] + stats_df["Negative"]).clip(0, 10) * stats_df["EloMod"] * stats_df["MinMod"] * stats_df["ScoreMod"]).clip(0, 10)

    # Selección de las columnas a mostrar
    stats_df = stats_df[["Player", "Match", "Rating"]]

    return stats_df

# --------------------------------------------------------------------------------------
# DEFINE RATINGS DE DELANTEROS
# --------------------------------------------------------------------------------------
def get_st_ratings(dict_positions_df: dict) -> pd.DataFrame:

    df = dict_positions_df["ST"].copy()

    # Obtenemos estadísticas
    stats_df = df[["Player","Match","Elo","OppElo","ScoreDiff","MinutesPlayed","Goals","ExpectedGoalsOnTarget","ShotAccuracy","BigChancesMissed",
                   "AerialWinRate","DuelWinRate","ContestWinRate","GoalAssists","KeyPasses","BigChancesCreated","BallRecoveries","WasFouled",
                   "HitWoodwork","Offsides","PenaltyMissed","PassAccuracy","YellowCards","RedCards"]]
    stats_df = stats_df.fillna(0)

    # Aplicamos normalización de columnas para aquellas contada
    stats_df["Goals"] = normalize_column(series=stats_df["Goals"], min_value=0, max_value=3)
    stats_df["ExpectedGoalsOnTarget"] = normalize_column(series=stats_df["ExpectedGoalsOnTarget"], min_value=0, max_value=2)
    stats_df["BigChancesMissed"] = normalize_column(series=stats_df["BigChancesMissed"], min_value=0, max_value=3)
    stats_df["GoalAssists"] = normalize_column(series=stats_df["GoalAssists"], min_value=0, max_value=3)
    stats_df["KeyPasses"] = normalize_column(series=stats_df["KeyPasses"], min_value=0, max_value=5)
    stats_df["BigChancesCreated"] = normalize_column(series=stats_df["BigChancesCreated"], min_value=0, max_value=4)
    stats_df["BallRecoveries"] = normalize_column(series=stats_df["BallRecoveries"], min_value=0, max_value=8)
    stats_df["WasFouled"] = normalize_column(series=stats_df["WasFouled"], min_value=0, max_value=6)
    stats_df["HitWoodwork"] = normalize_column(series=stats_df["HitWoodwork"], min_value=0, max_value=2)
    stats_df["Offsides"] = normalize_column(series=stats_df["Offsides"], min_value=0, max_value=5)
    stats_df["PenaltyMissed"] = normalize_column(series=stats_df["PenaltyMissed"], min_value=0, max_value=1)
    stats_df["YellowCards"] = normalize_column(series=stats_df["YellowCards"], min_value=0, max_value=1.5)
    stats_df["RedCards"] = normalize_column(series=stats_df["RedCards"], min_value=0, max_value=1)

    # Rating inicial
    stats_df["InitialRating"] = 6.0

    # Bonus positivo
    stats_df["Positive"] = (0.64 * stats_df["Goals"] + 0.40 * stats_df["ExpectedGoalsOnTarget"] + 0.40 * stats_df["ShotAccuracy"] +                    # Gol y remate: max +1.44
                            0.40 * stats_df["AerialWinRate"] + 0.32 * stats_df["DuelWinRate"] + 0.28 * stats_df["ContestWinRate"] +                   # Juego de pivote: max +1.00
                            0.40 * stats_df["GoalAssists"] + 0.24 * stats_df["KeyPasses"] + 0.16 * stats_df["BigChancesCreated"] +                   # Creación: max +0.80
                            0.20 * stats_df["BallRecoveries"] + 0.20 * stats_df["WasFouled"] +                                                        # Presión: max +0.40
                            0.50 * stats_df["HitWoodwork"] + 0.30 * stats_df["PassAccuracy"])                                                       # Bonus añadido

    # Penalizaciones
    stats_df["Negative"] = (-1.50 * stats_df["BigChancesMissed"] - 0.50 * stats_df["Offsides"] - 2.00 * stats_df["PenaltyMissed"] -
                            1.50 * stats_df["YellowCards"] - 4.00 * stats_df["RedCards"])

    # Ajuste por nivel del rival - ajuste +-5% por cada 400 puntos de diferencia
    stats_df["EloMod"] = 1 + 0.05*((stats_df["OppElo"] - stats_df["Elo"]) / 400)
    stats_df["MinMod"] = (stats_df["MinutesPlayed"] / 60).clip(0.5, 1)
    stats_df["ScoreMod"] = 1 + 0.1*stats_df["ScoreDiff"]

    # Cálculo del rating
    stats_df["Rating"] = ((stats_df["InitialRating"] + stats_df["Positive"] + stats_df["Negative"]).clip(0, 10) * stats_df["EloMod"] * stats_df["MinMod"] * stats_df["ScoreMod"]).clip(0, 10)

    # Selección de las columnas a mostrar
    stats_df = stats_df[["Player", "Match", "Rating"]]

    return stats_df

# --------------------------------------------------------------------------------------
# OBTENCIÓN DEL DATAFRAME DE RATINGS PARA TODOS LOS JUGADORES
# --------------------------------------------------------------------------------------
def get_ratings_df(dict_positions_df: dict) -> pd.DataFrame:

    # Obtenemos los dataframes con los ratings
    gk_df = get_gk_ratings(dict_positions_df=dict_positions_df)
    fb_df = get_fb_ratings(dict_positions_df=dict_positions_df)
    cb_df = get_cb_ratings(dict_positions_df=dict_positions_df)
    dm_df = get_dm_ratings(dict_positions_df=dict_positions_df)
    am_df = get_am_ratings(dict_positions_df=dict_positions_df)
    wg_df = get_wg_ratings(dict_positions_df=dict_positions_df)
    st_df = get_st_ratings(dict_positions_df=dict_positions_df)

    # Append de los dataframes y aplicamos nombre del jugador
    ratings_df = pd.concat([gk_df, fb_df, cb_df, dm_df, am_df, wg_df, st_df], ignore_index=True)
    ratings_df = ratings_df.sort_values(by="Player").reset_index(drop=True)

    return ratings_df

In [13]:
ratings_df = get_ratings_df(dict_positions_df=dict_positions_df)

---
## **3. Integración temporal de los ratings**

Las notas por partido se enriquecen y se agregan en el tiempo:

- **Ajuste por contexto competitivo.** Se normalizan el peso del torneo y el Elo y se construye un modificador acotado:

$$
\text{ContextMod} = \mathrm{clip}\big(0.85 + 0.10\cdot\text{TournWeightNorm} + 0.10\cdot\text{EloNorm},\; 0.85,\; 1.05\big)
$$

$$
\text{AdjustedRating} = \mathrm{clip}(\text{Rating}\cdot\text{ContextMod},\,0,\,10)
$$

- **Agregación bisemanal.** Para jugadores con **≥ 10 partidos**, las notas se promedian en periodos de **dos semanas** (`2W-MON`). Los periodos sin partidos se rellenan arrastrando el último valor (*forward fill*), lo que da una **serie temporal continua** del rendimiento de cada jugador (la que se dibuja en el gráfico de evolución de la web).

In [15]:
# --------------------------------------------------------------------------------------
# OBTENCIÓN DEL DATAFRAME DE RATINGS PARA TODOS LOS JUGADORES
# --------------------------------------------------------------------------------------
def get_match_rating_df(match_dataframe: pd.DataFrame, ratings_df: pd.DataFrame) -> pd.DataFrame:

    match_df = match_dataframe.copy()

    # Preparación de datos del partido que queremos
    match_data = prepare_match_data(match_dataframe=match_df)           # Usamos la función anterior
    match_data = match_data[["Match", "TournWeight", "Elo"]]            # Seleccionamos métricas de entorno para ponderar según tournament o elo

    # Añadimos la fecha del partido
    match_date_dict = dict(zip(match_df["ID"], match_df["Date"]))
    match_data["Date"] = match_data["Match"].map(match_date_dict)

    # Cruzamos con ratings_df
    match_rating_df = ratings_df.merge(match_data, on="Match", how="left").dropna().sort_values(by="Date")

    # Normalizamos contexto competitivo
    match_rating_df["TournWeightNorm"] = normalize_column(series=match_rating_df["TournWeight"], min_value=20, max_value=50)
    match_rating_df["EloNorm"] = normalize_column(series=match_rating_df["Elo"], min_value=500, max_value=2500)

    # Modificador contextual
    match_rating_df["ContextMod"] = (0.85 + 0.10 * match_rating_df["TournWeightNorm"] + 0.10 * match_rating_df["EloNorm"]).clip(0.85, 1.05)

    # Rating ajustado por contexto competitivo
    match_rating_df["AdjustedRating"] = (match_rating_df["Rating"] * match_rating_df["ContextMod"]).clip(0, 10)

    # Orden final y selección de columnas
    match_rating_df = match_rating_df.sort_values(by=["Player", "Date"]).reset_index(drop=True)
    match_rating_df = match_rating_df[["Player", "Match", "Date", "AdjustedRating"]]

    return match_rating_df

# --------------------------------------------------------------------------------------
# OBTENCIÓN DEL DATAFRAME DE RATINGS DE JUGADORES CADA DOS SEMANAS
# --------------------------------------------------------------------------------------
def biweek_players_ratings(match_rating_df: pd.DataFrame) -> pd.DataFrame:

    df = match_rating_df.copy()

    # Formato fecha
    df["Date"] = pd.to_datetime(df["Date"])

    # Filtramos jugadores que hayan jugado un mínimo de 10 partidos
    players_valid = (df.groupby("Player")["Match"].nunique().loc[lambda x: x >= 10].index)
    df = df[df["Player"].isin(players_valid)].copy()

    # Periodo cada dos semanas
    df["BiWeek"] = df["Date"].dt.to_period("2W-MON")

    # Agrupación cada 2 semanas
    player_biweek_rating_df = (df.groupby(["Player", "BiWeek"]).agg(PLAYED_MATCHES=("Match", "nunique"), mean_rating=("AdjustedRating", "mean")).reset_index())

    # Completar periodos vacíos
    def fill_missing_biweeks(player_df: pd.DataFrame) -> pd.DataFrame:

        player_id = player_df.name
        player_df = player_df.sort_values("BiWeek").copy()

        full_biweek_range = pd.period_range(start=player_df["BiWeek"].min(), end=player_df["BiWeek"].max(), freq="2W-MON")
        player_df = (player_df.set_index("BiWeek").reindex(full_biweek_range))

        player_df.index.name = "BiWeek"
        player_df["Player"] = player_id

        player_df["PLAYED_MATCHES"] = (player_df["PLAYED_MATCHES"].fillna(0).astype(int))
        player_df["mean_rating"] = player_df["mean_rating"].ffill()

        return player_df.reset_index()

    player_biweek_rating_df = (player_biweek_rating_df.groupby("Player", group_keys=False).apply(fill_missing_biweeks).reset_index(drop=True))

    # Conversión del periodo a fecha inicial
    player_biweek_rating_df["BiWeek"] = player_biweek_rating_df["BiWeek"].dt.start_time

    # Redondeo
    player_biweek_rating_df["mean_rating"] = (player_biweek_rating_df["mean_rating"].round(2))

    # Orden final
    player_biweek_rating_df = (player_biweek_rating_df.sort_values(["Player", "BiWeek"]).reset_index(drop=True))

    return player_biweek_rating_df

In [16]:
# Obtención del math rating de cada jugador
match_rating_df = get_match_rating_df(match_dataframe=MATCH.copy(), ratings_df=ratings_df.copy())

# Obtención del dataframe agregado cada dos semanas de rating por jugador
biweek_rating_df = biweek_players_ratings(match_rating_df=match_rating_df.copy())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_47732\1925544111.py:72: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_biweek_rating_df = (player_biweek_rating_df.groupby("Player", group_keys=False).apply(fill_missing_biweeks).reset_index(drop=True))


---

## **4. Valoración del jugador (Rating 65–95)**

A partir del rendimiento **reciente** (desde el inicio de la temporada) se calcula una valoración tipo videojuego.

In [24]:
# --------------------------------------------------------------------------------------
# OBTENCIÓN DEL DATAFRAME CON LA MEDIA DE CADA JUGADOR
# --------------------------------------------------------------------------------------
def obtain_player_value_df(match_data: pd.DataFrame, players_df: pd.DataFrame, player_stats: pd.DataFrame, team_df: pd.DataFrame, biweek_rating_df: pd.DataFrame) -> pd.DataFrame:

    # Obtención del inicio de temporada
    today = datetime.now()
    if today.month >= 9:
        season_year = today.year
    else:
        season_year = today.year - 1
    last_months = pd.Timestamp(f"{season_year}-08-01")

    # Obtención de un diccionario que nos muestra la normalización por 90 de los jugadores
    def obtain_dict_per90(match_data: pd.DataFrame, player_stats: pd.DataFrame, last_months) -> dict:

        # Merged dataframe y selección de columnas
        match_data = match_data.rename(columns={"ID":"Match"})
        merged_df = player_stats.merge(match_data, on="Match", how="inner")
        merged_df = merged_df[["Match", "Player", "MinutesPlayed", "Date"]]

        # Filtrado por fecha
        merged_df["Date"] = pd.to_datetime(merged_df["Date"])
        merged_df = merged_df[merged_df["Date"] >= last_months]

        # Factor por 90 y media por jugador
        merged_df["Per90Factor"] = merged_df["MinutesPlayed"] / 90
        player_per90factor_df = (merged_df.groupby("Player", as_index=False).agg(mean_per90factor=("Per90Factor", "mean")))

        # Diccionario
        dict_per90_factor = dict(zip(player_per90factor_df["Player"], player_per90factor_df["mean_per90factor"]))

        return dict_per90_factor

    # Obtención de un diccionario con el ELO del equipo normalizado
    def obtain_dict_elo(team_df: pd.DataFrame, players_df: pd.DataFrame) -> dict:

        # Obtenemos elo por equipo
        team_df = team_df[["ID", "EloRating"]].copy()

        elo_range = team_df["EloRating"].max() - team_df["EloRating"].min()

        if elo_range == 0:
            team_df["EloRatingNorm"] = 0.5
        else:
            team_df["EloRatingNorm"] = (0.5 + ((team_df["EloRating"] - team_df["EloRating"].min()) / elo_range) * (1 - 0.5))

        # Buscamos jugadores y club y aplicamos equipo
        players_df = players_df[["ID", "Name", "Team"]].copy()
        players_club_df = players_df.merge(team_df, left_on="Team", right_on="ID", how="inner")

        # Return del diccionario
        return dict(zip(players_club_df["ID_x"], players_club_df["EloRatingNorm"]))

    # Diccionario de mapeo según la importancia de los partidos jugados
    def obtain_dict_match_importance(match_data: pd.DataFrame, player_stats: pd.DataFrame, last_months) -> dict:

        # Merged dataframe y selección de columnas
        match_data = match_data.rename(columns={"ID":"Match"})
        merged_df = player_stats.merge(match_data, on="Match", how="inner")
        merged_df = merged_df[["Match", "Player", "MinutesPlayed", "Date", "League"]]

        # Filtrado por fecha
        merged_df["Date"] = pd.to_datetime(merged_df["Date"])
        merged_df = merged_df[merged_df["Date"] >= last_months]

        # Añadimos peso de tournament
        tournament_k = {"la_liga": 40, "premier_league": 40, "bundesliga": 40, "serie_a": 40, "ligue_1": 40, "champions_league": 50, "europa_league": 30,
                        "copa_del_rey": 30, "la_liga_hypermotion": 20, "championship": 20, "eredivise": 30, "liga_portugal": 30, "conference_league": 30,
                        "copa_libertadores": 20, "liga_mx": 20, "serie_a_brazil": 20, "liga_profesional": 20, "first_division_a": 20, "major_league_soccer": 20,
                        "saudi_pro_league": 20, "super_lig": 20, "superligaen": 20, "swiss_super_league": 20, "allvenskan": 20, "eliteserien": 20,
                        "conmebol_sudamericana": 20}
        merged_df["TournamentWeight"] = merged_df["League"].map(tournament_k)

        # Obtenemos métricas por jugador
        player_tournament_weight_df = (merged_df.groupby("Player", as_index=False).agg(MeanTournamentWeight=("TournamentWeight", "mean"),
                                                                                    MaxTournamentWeight=("TournamentWeight", "max"),
                                                                                    NumCompetitions=("League", "nunique"),
                                                                                    MatchesPlayed=("Match", "nunique"),
                                                                                    TotalMinutes=("MinutesPlayed", "sum")))

        # Score bruto: nivel medio * factor diversidad
        player_tournament_weight_df["CompetitionScore"] = (player_tournament_weight_df["MeanTournamentWeight"] * (1 + 0.10 * (player_tournament_weight_df["NumCompetitions"] - 1)))

        # Normalizamos
        min_score = player_tournament_weight_df["CompetitionScore"].min()
        max_score = player_tournament_weight_df["CompetitionScore"].max()
        score_range = max_score - min_score

        if score_range == 0:
            player_tournament_weight_df["NormWeight"] = 0.5
        else:
            player_tournament_weight_df["NormWeight"] = (0.5 + ((player_tournament_weight_df["CompetitionScore"] - min_score) / score_range) * 0.5)

        return dict(zip(player_tournament_weight_df["Player"], player_tournament_weight_df["NormWeight"]))

    # Aplicamos funciones
    dict_per90 = obtain_dict_per90(match_data=match_data, player_stats=player_stats, last_months=last_months)
    dict_elo = obtain_dict_elo(team_df=team_df, players_df=players_df)
    dict_match_imp = obtain_dict_match_importance(match_data=match_data, player_stats=player_stats, last_months=last_months)

    # Obtención del dataframe con el rating medio por jugador
    player_rating_base_df = (biweek_rating_df[biweek_rating_df["BiWeek"] >= last_months].groupby("Player", as_index=False).agg(Rating=("mean_rating", "mean"), Matches=("PLAYED_MATCHES", "sum")))

    # Ajuste bayesiano del rating para evitar sobrevalorar jugadores con pocos partidos
    global_rating = (biweek_rating_df[biweek_rating_df["BiWeek"] >= last_months]["mean_rating"].mean())
    player_rating_base_df["AdjustedRating"] = ((player_rating_base_df["Matches"] * player_rating_base_df["Rating"]) + (8 * global_rating)) / (player_rating_base_df["Matches"] + 8)
    last_rating_by_player_df = player_rating_base_df[["Player", "AdjustedRating"]].copy()
    last_rating_by_player_df.columns = ["Player", "Rating"]

    # Aplicamos normalización del rating por normalización de minutos
    last_rating_by_player_df["Per90Factor"] = last_rating_by_player_df["Player"].map(dict_per90)
    last_rating_by_player_df["EloFactor"] = last_rating_by_player_df["Player"].map(dict_elo)
    last_rating_by_player_df["MatchImpFactor"] = last_rating_by_player_df["Player"].map(dict_match_imp)

    # # Buscamos posición por jugador
    player_pos_dict = dict(zip(players_df["ID"], players_df["FirstPos"]))
    last_rating_by_player_df["Position"] = last_rating_by_player_df["Player"].map(player_pos_dict)
    last_rating_by_player_df = last_rating_by_player_df.dropna(subset="Position")

    # Obtención del dataframe con todos los datos
    player_rating_df = last_rating_by_player_df[["Player","Position","Rating","Per90Factor","EloFactor","MatchImpFactor"]].copy()

    # Cálculo del rating final - normalización inicial
    player_rating_df["RatingNorm"] = player_rating_df["Rating"] / 10
    player_rating_df["MediaRaw"] = (0.65 * player_rating_df["RatingNorm"] + 0.15 * player_rating_df["Per90Factor"] + 0.10 * player_rating_df["EloFactor"] + 0.10 * player_rating_df["MatchImpFactor"])
    player_rating_df["Rating"] = (65 + player_rating_df["MediaRaw"] * (95 - 65)).round(0)

    # Añadimos el nombre del jugador
    player_name_dict = dict(zip(players_df["ID"], players_df["Name"]))
    player_rating_df["Name"] = player_rating_df["Player"].map(player_name_dict)

    # Selección de columnas
    player_rating_df = (player_rating_df[["Player", "Name", "Position", "Rating", "Per90Factor", "EloFactor", "MatchImpFactor"]].sort_values(by="Rating", ascending=False))

    return player_rating_df

In [27]:
players_value_df = obtain_player_value_df(match_data=MATCH.copy(), players_df=PLAYER.copy(), player_stats=STATS_PLAYER.copy(), team_df=TEAM.copy(), biweek_rating_df=biweek_rating_df)
display(players_value_df.head(20))

,Player,Name,Position,Rating,Per90Factor,EloFactor,MatchImpFactor
3069,JW1K2,David Raya,GK,90.0,1.000000,0.987971,0.860494
4281,RM6JV,Manuel Neuer,GK,90.0,0.960000,1.000000,0.870000
369,2F5HQ,Pau Cubarsí,CB,90.0,0.927160,0.978269,0.903704
5469,ZDKUF,Joan García,GK,90.0,0.997832,0.978269,0.901084
297,1YCGK,Gianluigi Donnarumma,GK,90.0,1.000000,0.950555,0.865310
3769,OG5YX,Manuel Akanji,CB,89.0,0.913072,0.936246,0.869281
4867,VI9RS,Declan Rice,AM,89.0,0.946253,0.987971,0.858204
5162,XBPMR,Gonçalo Inácio,CB,89.0,0.944444,0.909083,0.747396
1316,8L3FA,Jonathan Tah,CB,89.0,0.902867,1.000000,0.877688
2830,IBWQB,Jakub Kiwior,CB,89.0,0.988462,0.906624,0.680556


---

## **5. Cálculo del potencial (Potential 65–99)**

El potencial estima la **evolución futura** del jugador. Su ingrediente diferencial es la **edad**, mediante una curva decreciente (cuanto más joven, más margen de mejora):

| Edad | AgePotential |
|:---:|:---:|
| ≤ 18 | 1.00 |
| ≤ 20 | 0.90 |
| ≤ 22 | 0.80 |
| ≤ 24 | 0.65 |
| ≤ 26 | 0.45 |
| ≤ 29 | 0.25 |
| ≤ 31 | 0.10 |
| > 31 | 0.00 |

Se añade un **factor de muestra** (más partidos = más fiabilidad): $\text{SampleFactor} = \mathrm{clip}(\text{PartidosJugados}/20,\,0,\,1)$.

El potencial bruto combina el nivel actual con el margen por edad, contexto y fiabilidad:

$$
\text{PotentialRaw} = 0.45\cdot\text{MediaNorm} + 0.25\cdot\text{AgePotential} + 0.10\cdot\text{Per90} + 0.10\cdot\text{Elo} + 0.05\cdot\text{MatchImp} + 0.05\cdot\text{SampleFactor}
$$

donde $\text{MediaNorm} = \mathrm{clip}((\text{Rating} - 65)/30,\,0,\,1)$. La escala final es **65–99**:

$$
\text{Potential} = \mathrm{round}\big(65 + \text{PotentialRaw}\cdot(99 - 65)\big)
$$

In [34]:
# --------------------------------------------------------------------------------------
# OBTENCIÓN DEL DATAFRAME CON EL POTENCIAL DEL JUGADOR
# --------------------------------------------------------------------------------------
def obtain_player_potential_df(player_df: pd.DataFrame, match_data: pd.DataFrame, player_stats:pd.DataFrame, players_value_df: pd.DataFrame) -> pd.DataFrame:

    # Obtención del inicio de temporada
    today = datetime.now()
    if today.month >= 9:
        season_year = today.year
    else:
        season_year = today.year - 1
    last_months = pd.Timestamp(f"{season_year}-08-01")

    # Obtención de la fecha de nacimiento del jugador y transformación a datetime
    player_date_birth = player_df[["ID", "DateBirth"]].dropna()
    player_date_birth["DateBirth"] = pd.to_datetime(player_date_birth["DateBirth"], errors="coerce")
    player_date_birth["Age"] = ((today - player_date_birth["DateBirth"]).dt.days / 365.25)

    # Factor de potencial por edad
    def get_age_potential(age: float) -> float:
        if pd.isna(age):
            return 0.50
        if age <= 18:
            return 1.00
        elif age <= 20:
            return 0.90
        elif age <= 22:
            return 0.80
        elif age <= 24:
            return 0.65
        elif age <= 26:
            return 0.45
        elif age <= 29:
            return 0.25
        elif age <= 31:
            return 0.10
        else:
            return 0.00
        
    player_date_birth["AgePotential"] = player_date_birth["Age"].apply(get_age_potential)

    # Obtención de los minutos jugados por jugador
    def dict_matches_played(match_data: pd.DataFrame, player_stats: pd.DataFrame) -> dict:

        # Merged dataframe y selección de columnas
        match_data = match_data.rename(columns={"ID":"Match"})
        merged_df = player_stats.merge(match_data, on="Match", how="inner")
        merged_df = merged_df[["Match", "Player", "Date"]]

        # Filtrado por fecha
        merged_df["Date"] = pd.to_datetime(merged_df["Date"])
        merged_df = merged_df[merged_df["Date"] >= last_months]

        # Número de partidos jugados por jugador
        player_matches_df = (merged_df.groupby("Player", as_index=False).agg(MatchesPlayed=("Match", "nunique")))

        # Diccionario
        dict_matches_played = dict(zip(player_matches_df["Player"],player_matches_df["MatchesPlayed"]))
        return dict_matches_played

    # Aplicamos los partidos jugados para cada jugador
    players_potential_df = players_value_df.copy()
    players_potential_df["PlayedMatches"] = players_potential_df["Player"].map(dict_matches_played(match_data=match_data, player_stats=player_stats))

    # Factor de muestra: más partidos = más fiabilidad
    players_potential_df["SampleFactor"] = (players_potential_df["PlayedMatches"] / 20).clip(0, 1)

    # Cruzamos para tener el potencial de edad
    player_date_birth = player_date_birth.rename(columns={"ID":"Player"})
    players_potential_df = players_potential_df.merge(player_date_birth, on="Player", how="inner")

    # Normalización de la media actual
    players_potential_df["MediaNorm"] = ((players_potential_df["Rating"] - 65) / (95 - 65)).clip(0, 1)

    # Score bruto de potencial
    players_potential_df["PotentialRaw"] = (0.45 * players_potential_df["MediaNorm"] + 0.25 * players_potential_df["AgePotential"] + 0.10 * players_potential_df["Per90Factor"] + 0.10 * players_potential_df["EloFactor"] +
                                            0.05 * players_potential_df["MatchImpFactor"] + 0.05 * players_potential_df["SampleFactor"])

    # Escala final 65-99
    players_potential_df["Potential"] = (65 + players_potential_df["PotentialRaw"] * (99 - 65)).round(0)

    return players_potential_df.reset_index(drop=True)

In [36]:
player_potential_df = obtain_player_potential_df(player_df=PLAYER.copy(), match_data=MATCH.copy(), player_stats=STATS_PLAYER.copy(), players_value_df=players_value_df)
display(player_potential_df.sort_values(by="Potential", ascending=False).head(10))

,Player,Name,Position,Rating,Per90Factor,EloFactor,MatchImpFactor,PlayedMatches,SampleFactor,DateBirth,Age,AgePotential,MediaNorm,PotentialRaw,Potential
2,2F5HQ,Pau Cubarsí,CB,90.0,0.927160,0.978269,0.903704,45,1.0,2007-01-22,19.422313,0.9,0.833333,0.885728,95.0
25,UUWLQ,Warren Zaïre-Emery,AM,89.0,0.905247,0.967352,0.880015,36,1.0,2006-03-08,20.298426,0.8,0.800000,0.841261,94.0
114,CJX5H,Lamine Yamal,RW,88.0,0.924661,0.978269,0.901084,41,1.0,2007-07-13,18.951403,0.9,0.766667,0.855347,94.0
19,3DDAV,Jacobo Ramón,CB,89.0,0.959111,0.913561,0.777778,25,1.0,2005-01-06,21.464750,0.8,0.800000,0.836156,93.0
229,1TYYS,Luka Vušković,CB,87.0,1.000000,0.831671,0.777778,23,1.0,2007-02-24,19.331964,0.9,0.733333,0.827056,93.0
319,H0GAT,Tylel Tati,CB,87.0,0.984568,0.777255,0.777778,18,0.9,2008-01-19,18.431211,0.9,0.733333,0.815071,93.0
163,TC9KZ,Nico Paz,AM,88.0,0.913580,0.913561,0.777778,27,1.0,2004-09-09,21.790554,0.8,0.766667,0.816603,93.0
160,VUHLR,Jon Martin,CB,88.0,1.000000,0.851994,0.807870,30,1.0,2006-04-23,20.172485,0.8,0.766667,0.820593,93.0
134,13QZF,Kenan Yıldız,LW,88.0,0.891049,0.891660,0.871528,36,1.0,2005-05-04,21.141684,0.8,0.766667,0.816847,93.0
131,3Z54G,Robin Risser,GK,88.0,1.000000,0.897668,0.777778,26,1.0,2004-12-02,21.560575,0.8,0.766667,0.823656,93.0


---

## **6. Función principal de aplicado**

Una función *wrapper* encadena todo el proceso (preparación → rating por partido → integración temporal → Rating → Potential) y mapea los valores finales a cada jugador.

In [37]:
from typing import Tuple

def main_players_rating(match_df: pd.DataFrame, player_info_df: pd.DataFrame, stats_player_df: pd.DataFrame, team_info_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:

    # Borramos columnas de rating y de potencial a los dataframes que vamos a modificar
    if "Rating" in stats_player_df.columns:
        stats_player_df = stats_player_df.drop(columns="Rating")
    if "Rating" in player_info_df.columns and "Potential" in player_info_df.columns:
        player_info_df = player_info_df.drop(columns=["Rating", "Potential"])

    # Preparación de datos
    dict_positions_df = prepare_stats_player(stats_dataframe=stats_player_df.copy(), match_dataframe=match_df.copy())

    # Dataframe con los ratings de los jugadores por partido
    ratings_df = get_ratings_df(dict_positions_df=dict_positions_df)

    # Obtención del match rating de cada jugador
    match_rating_df = get_match_rating_df(match_dataframe=match_df.copy(), ratings_df=ratings_df.copy())

    # Obtención del dataframe agregado cada dos semanas de rating por jugador
    biweek_rating_df = biweek_players_ratings(match_rating_df=match_rating_df.copy())

    # Obtención de la media de los jugadores
    players_value_df = obtain_player_value_df(match_data=match_df.copy(), players_df=player_info_df.copy(), player_stats=stats_player_df.copy(), team_df=team_info_df.copy(), biweek_rating_df=biweek_rating_df)

    # Obtención del potencial de jugadores
    player_potential_df = obtain_player_potential_df(player_df=player_info_df.copy(), match_data=match_df.copy(), player_stats=stats_player_df.copy(), players_value_df=players_value_df)

    # Hacemos merge con los partidos y el rating obtenido para cada jugador en aquel partido
    stats_players_with_rating = stats_player_df.merge(ratings_df.drop_duplicates(subset=["Player", "Match"]), on=["Player", "Match"], how="left")

    # Aplicamos el rating y el potencial a la tabla de datos de información de jugadores - obtenemos diccionarios
    dict_rating = dict(zip(player_potential_df["Player"], player_potential_df["Rating"]))
    dict_potential = dict(zip(player_potential_df["Player"], player_potential_df["Potential"]))

    # Aplicamos diccionarios
    player_info_df["Rating"] = player_info_df["ID"].map(dict_rating)
    player_info_df["Potential"] = player_info_df["ID"].map(dict_potential)

    return stats_players_with_rating, player_info_df

In [ ]:
stats_players_with_rating, player_info_df = main_players_rating(match_df=MATCH.copy(), player_info_df=PLAYER.copy(), stats_player_df=STATS_PLAYER.copy(), team_info_df=TEAM.copy())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_47732\1925544111.py:72: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  player_biweek_rating_df = (player_biweek_rating_df.groupby("Player", group_keys=False).apply(fill_missing_biweeks).reset_index(drop=True))


In [43]:
display(player_info_df.dropna(subset=["Rating", "Potential"])[["Name", "Rating", "Potential"]].sort_values(by=["Rating", "Potential"], ascending=False).reset_index(drop=True).head(50))

,Name,Rating,Potential
0,Pau Cubarsí,90.0,95.0
1,Joan García,90.0,92.0
2,Gianluigi Donnarumma,90.0,90.0
3,David Raya,90.0,89.0
4,Manuel Neuer,90.0,88.0
5,Warren Zaïre-Emery,89.0,94.0
6,Jacobo Ramón,89.0,93.0
7,Jonas Urbig,89.0,92.0
8,Lucas Chevalier,89.0,91.0
9,Marc Guéhi,89.0,91.0


---